<a href="https://colab.research.google.com/github/Amruth-U-tech/DL-Journey/blob/main/08-LLMs/LLM-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Set model to GPT-2
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("using model:", model_name)
print("device:", device)

def show_next_token_predictions(sentence: str, top_k: int = 5):
    # Tokenize input
    inputs = tokenizer(sentence, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Get logits for the last token in the sequence
    next_token_logits = outputs.logits[0, -1, :]

    # Calculate probabilities
    probs = torch.softmax(next_token_logits, dim=-1)
    topk = torch.topk(probs, k=top_k)

    print(f"input sentence: {sentence}")
    print("\ntop predictions for next token:")
    for rank in range(top_k):
        token_id = topk.indices[rank].item()
        token_str = tokenizer.decode([token_id])
        prob = topk.values[rank].item()
        print(f"{rank+1:>2}. '{token_str}'".ljust(20) + f" (prob = {prob:.3f})")

# Example usage
sentence = "The capital of France is"
show_next_token_predictions(sentence, top_k=5)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

using model: gpt2
device: cpu
input sentence: The capital of France is

top predictions for next token:
 1. ' the'           (prob = 0.085)
 2. ' now'           (prob = 0.048)
 3. ' a'             (prob = 0.046)
 4. ' France'        (prob = 0.032)
 5. ' Paris'         (prob = 0.032)


### Generate a Full Sentence
To generate multiple tokens, we can use the `generate` method. This allows us to specify the maximum length and sampling strategies.

In [4]:
def generate_text(prompt, max_new_tokens=20, temperature=0.7):
    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate output tokens
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,           # Set to True for creative text, False for greedy
            temperature=temperature,  # Higher = more random
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode and print results
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated_text}")

# Try generating a longer full sentence
generate_text("The capital of France is", max_new_tokens=50)

Prompt: The capital of France is
Generated: The capital of France is on the French border, and yet we have to move from there. The French people are so happy to have a safe haven for refugees. They're very sad about the fact that we're not able to send them out of France. We're being


### Comparing Greedy vs. Sampling
Greedy search always picks the token with the highest probability, which is usually better for factual queries. Sampling (with temperature) is better for creative writing but can lead to the 'vague' or 'inaccurate' results you observed.

In [6]:
def compare_generation_methods(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # 1. Greedy Search with Repetition Penalty
    penalty_output = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        repetition_penalty=1.5, # Penalty for repetition
        pad_token_id=tokenizer.eos_token_id
    )

    # 2. Standard Greedy Search (for comparison)
    greedy_output = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    print(f"Prompt: {prompt}")
    print(f"--- Standard Greedy: {tokenizer.decode(greedy_output[0], skip_special_tokens=True)}")
    print(f"--- With Repetition Penalty (1.5): {tokenizer.decode(penalty_output[0], skip_special_tokens=True)}")

compare_generation_methods("The capital of France is")

Prompt: The capital of France is
--- Standard Greedy: The capital of France is the capital of the French Republic, and the capital of the French Republic is the capital of the French Republic.

The French Republic is the capital
--- With Repetition Penalty (1.5): The capital of France is the city, and its inhabitants are not only rich but also poor. The French have a long history in this country; they were once called "the
